In [1]:
%cd ../
%ls

/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/venv/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


/Users/delapazm/Desktop/wellcome_academic_graph_toolkit
CH_authors/                   dist/
CH_coauthors_top/             edges_all.json
CH_coauthors_v2/              environment.yml
CH_coauthors_wt/              geographies/
CH_coauthors_wt.zip           idr/
CH_coauthors_wt_lmic/         nodes_all.json
CH_coauthors_wt_lmic_all/     pyproject.toml
CH_coauthors_wt_lmic_all.zip  requirements.txt
Makefile                      venv/
README.md                     wag_toolkit/
careers/


In [15]:
from wag_toolkit.locations import Locations
import pandas as pd
import json
from collections import Counter

In [16]:
countries_class = pd.read_excel('geographies/CH_files/CLASS.xlsx', sheet_name='List of economies')
lmic_list = countries_class[countries_class['Income group'].isin(['Low income', 'Lower middle income', 'Upper middle income'])]
lmic_list.head()

,Economy,Code,Region,Income group,Lending category
0,Afghanistan,AFG,South Asia,Low income,IDA
1,Albania,ALB,Europe & Central Asia,Upper middle income,IBRD
2,Algeria,DZA,Middle East & North Africa,Upper middle income,IBRD
5,Angola,AGO,Sub-Saharan Africa,Lower middle income,IBRD
7,Argentina,ARG,Latin America & Caribbean,Upper middle income,IBRD


In [ ]:
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Only Wellcome
candh_intersection = pd.read_excel('geographies/CH_files/C&HPublicationExtraction.xlsx', sheet_name='Grant Awards')


# All C&H
# candh_intersection = pd.read_parquet('geographies/CH_files/climate_health_intersection_just_ids.parquet')

candh_intersection.head()

,Rank,Manually Tagged,Publication ID,DOI,PMID,PMCID,ISBN,Title,Abstract,Acknowledgements,...,Fields of Research (ANZSRC 2020),RCDC Categories,HRCS HC Categories,HRCS RAC Categories,Health Research Areas,Broad Research Areas,Cancer Types,CSO Categories,Units of Assessment,Sustainable Development Goals
0,500,No,pub.1170152051,10.1126/sciadv.adj3832,38536907.0,PMC10971398,NaN,Food matters: Dietary shifts increase the feas...,A transition to healthy diets such as the EAT-...,Funding: This work received funding from the E...,...,32 Biomedical and Clinical Sciences; 3210 Nutr...,Nutrition; Prevention,Cancer; Cardiovascular; Metabolic and endocrin...,NaN,NaN,Public Health,NaN,NaN,B07 Earth Systems and Environmental Sciences,13 Climate Action
1,300,No,pub.1160572336,10.1016/s0140-6736(23)01290-4,37442146.0,NaN,NaN,EAT–Lancet Commission 2.0: securing a just tra...,NaN,The EAT–Lancet Commission 2.0 is made possible...,...,32 Biomedical and Clinical Sciences; 42 Health...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100,No,pub.1170161790,10.1016/j.ecolecon.2024.108187,NaN,NaN,NaN,Peatland restoration in Germany: A dynamic gen...,Drained peatland currently contributes 7.5% of...,Declaration of competing interest The authors ...,...,38 Economics; 3801 Applied Economics,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"A06 Agriculture, Veterinary and Food Science",13 Climate Action; 15 Life on Land
3,500,No,pub.1165965381,10.3390/insects14110875,37999075.0,PMC10671961,NaN,Bee Assemblage in the Southern Chihuahuan Dese...,Recognizing how populations fluctuate over tim...,We acknowledge Consejo Nacional de Ciencia y T...,...,31 Biological Sciences; 3103 Ecology; 3109 Zoo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A05 Biological Sciences,15 Life on Land
4,500,No,pub.1151121272,10.1038/s43016-022-00588-7,37118149.0,NaN,NaN,Global food systems transitions have enabled a...,"Over the past 50 years, food systems worldwide...",We thank the following colleagues for their co...,...,"30 Agricultural, Veterinary and Food Sciences;...",Health Disparities; Nutrition; Social Determin...,Generic health relevance,NaN,Population & Society,NaN,NaN,NaN,C22 Anthropology and Development Studies,2 Zero Hunger


In [18]:
pub_ids = list(set(candh_intersection['Publication ID'].tolist()))

In [19]:
dummy_query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication) RETURN * LIMIT 1"""
loc = Locations(dummy_query)

query = """MATCH (r:Researcher)-[a:AUTHORED]->(p:Publication)
            WHERE p.dimensions_publication_id IN {}
            RETURN p.dimensions_publication_id AS dimensions_publication_id, p.year AS year,
                a.institutions AS grid_id"""
loc.lookup_query(query=query, lookup=pub_ids)

100%|██████████| 1/1 [00:00<00:00,  2.06it/s]


In [20]:
data =pd.DataFrame(loc.data)
data = data.explode("grid_id").dropna()
data.head()

,dimensions_publication_id,year,grid_id
8,pub.1146763677,2022.0,grid.189504.1
9,pub.1146763677,2022.0,grid.8991.9
10,pub.1117408551,2019.0,grid.4991.5
11,pub.1117408551,2019.0,grid.4991.5
12,pub.1117408551,2019.0,grid.4991.5


In [21]:
loc._clean_grid_ids()
loc.extract_edges()
loc.extract_locations("country")

loc.convert_edges()
loc.calculate_adjacency_matrices()

100%|██████████| 1/1 [00:00<00:00,  9.71it/s]
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df = pd.crosstab(
/Users/delapazm/Desktop/wellcome_academic_graph_toolkit/wag_toolkit/locations.py:183: FutureWarning: The provided callable <built-in function sum> is currently using DataFrameGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep cu

2022.0
2019.0
2020.0
2021.0
2023.0
2018.0
2017.0


In [22]:
import numpy as np
loc.adjacency_matrices = {np.int64(key): value for key, value in loc.adjacency_matrices.items()}

In [23]:
lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
lmic_list['Economy'] = lmic_list['Economy'].replace("Iran, Islamic Rep.", "Iran")

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_64437/708080782.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Côte d’Ivoire", "Ivory Coast")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_64437/708080782.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  lmic_list['Economy'] = lmic_list['Economy'].replace("Gambia, The", "Gambia")
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_64437/708080782.py:3: SettingWithCopyW

In [24]:
# Only account for LMIC
only_lmic = True

if only_lmic:
    for year in loc.adjacency_matrices:
        for country in loc.adjacency_matrices[year]['All'].index:
            if country == 'All' or country == 'total':
                continue
            for country2 in loc.adjacency_matrices[year]['All'][country].index:
                if country in list(lmic_list['Economy']) or country2 in list(lmic_list['Economy']):
                    continue
                else:
                    loc.adjacency_matrices[year]['All'][country][country2]=0

/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykernel_64437/191586841.py:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  loc.adjacency_matrices[year]['All'][country][country2]=0
/var/folders/dy/p3vg7l610wx7yc4y2908psy00000gp/T/ipykern

In [25]:
loc.load_visjs_nodes_and_edges(node_scaling=0.0015, edge_scaling=0.1, directed=False, threshold=0, font = {"size": 20, "face": "Helvetica Neue"}, node_count='total')

In [26]:
dirname = 'CH_authors'
loc.to_visjs(vis_name="locations", directed=False, template='geographies/locations.html', dirname=dirname)
loc._to_json("nodes_all.json", loc.vis_nodes)
loc._to_json("edges_all.json", loc.vis_edges)

In [27]:
import os
import shutil

with(open(f'{dirname}/edges.json','r')) as f:
    edges = json.load(f)

os.rename(f'{dirname}/nodes.json', f'{dirname}/nodes_all.json')
shutil.copyfile('geographies/positions.json', f'{dirname}/positions.json')

def edges_to_dict(edges):
    edge_dict = {}
    for e in edges:
        year = e['year']
        _ = e.pop('year')
        if edge_dict.get(year):
            edge_dict[year].append(e)
        else:
            edge_dict[year] = [e]
    return edge_dict


with(open(f'{dirname}/dict_edges_all.json','w')) as f:
        json.dump(edges_to_dict(edges), f)